# Phase 4 — Confirmatory run (PREREG v2), in ~4-hour sessions

**When do conformal hallucination guarantees survive distribution shift?**
The code is cloned at the git tag **`prereg-v2`** (frozen). This notebook only runs it.

**Kaggle settings:** Accelerator **GPU T4 x2**, **Internet ON**.

**Session 1**
1. Run cells 1–5 interactively (setup + smoke test, ~25 min). Cell 5 must print captions from all three VLMs and some detector scores.
2. **Save Version → Save & Run All (Commit)**. The run stops by itself after `SESSION_HOURS` (cell 6, default 3.5 h) and saves everything finished.

**Sessions 2, 3, … (until cell 7 says COMPLETE)**
1. Open the notebook → right panel **Add Input → Your Work → this notebook** (choose the latest version; its Output contains all finished work).
2. **Save Version → Save & Run All (Commit)**. Cell 4 imports the finished work, the smoke test is skipped, and only the rest runs.
3. After each session, open the version's log: cell 7 prints the progress (e.g. `captions 100%, owl 62%`) and **COMPLETE** at the end.

When COMPLETE: download `BCRBP_phase4/phase4_results.zip` from that version's Output tab and put it in Drive → `BCRBP_phase2`.

Nothing is lost between sessions: every caption and score is saved the moment it is computed.

In [ ]:
# 1. Environment and GPUs
import os, subprocess, sys, time
ON_KAGGLE = os.path.exists("/kaggle/working")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU FOUND - switch the accelerator to GPU T4 x2")
import torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU)

In [ ]:
# 2. Folders
WORK = "/kaggle/working" if ON_KAGGLE else os.getcwd()
OUT = os.path.join(WORK, "BCRBP_phase4")
SMOKE = os.path.join(WORK, "BCRBP_phase4_smoke")
os.makedirs(OUT, exist_ok=True)
print("Results go to:", OUT)

In [ ]:
# 3. Code at the pre-registered tag, and packages
REPO = "https://github.com/antonyjasfer/Budget-Coupled-Robust-Belief-Propagation-for-Visual-Hallucination-Detection.git"
TAG = "prereg-v2"
os.chdir(WORK)
if not os.path.exists("bcrbp"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", TAG, REPO, "bcrbp"], check=True)
os.chdir(os.path.join(WORK, "bcrbp"))
!pip -q install inflect bitsandbytes accelerate gdown
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!git log --oneline -1

In [ ]:
# 4. Setup, images, and import of any earlier session's work (CPU, ~10-15 min)
!python scripts/phase4.py setup  --out "$OUT"
!python scripts/phase4.py import --out "$OUT"
!python scripts/phase4.py images --out "$OUT"

In [ ]:
# 5. Smoke test in a separate folder (2 images per set, ~10-15 min). Skipped in later sessions.
import shutil, json, glob
already = glob.glob(os.path.join(OUT, "*.s*.jsonl"))
if already:
    print(f"Skipping the smoke test: {len(already)} result files imported from an earlier session.")
else:
    os.makedirs(os.path.join(SMOKE, "amber"), exist_ok=True)
    for f in ["synonyms.txt", "prep.json"]:
        shutil.copy(os.path.join(OUT, f), os.path.join(SMOKE, f))
    shutil.copy(os.path.join(OUT, "amber", "annotations.json"), os.path.join(SMOKE, "amber", "annotations.json"))
    for src, dst in [(os.path.join(OUT, "images"), os.path.join(SMOKE, "images")),
                     (os.path.join(OUT, "amber", "images"), os.path.join(SMOKE, "amber", "images"))]:
        if not os.path.exists(dst):
            os.symlink(src, dst)
    !python scripts/phase4.py gpu --out "$SMOKE" --limit 2
    show = lambda stem: [json.loads(l) for p in glob.glob(f"{SMOKE}/{stem}.s*.jsonl") for l in open(p) if l.strip()]
    for stem in ["captions_llava_coco", "captions_llava_amber", "captions_qwen", "captions_internvl"]:
        rows = show(stem)
        print(f"\n{stem}: {len(rows)} rows", "|", (rows[0]["caption"][:200] if rows else "MISSING"))
    print("\nowl rows:", len(show("owl")), "| swap rows:", len(show("swap")))

In [ ]:
# 6. GPU work for THIS session: one process per GPU, stopped cleanly after SESSION_HOURS.
SESSION_HOURS = 3.5   # leaves ~30 min for setup, progress check and saving (about 4 h per session)
procs = []
for k in range(max(1, NGPU)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(os.path.join(OUT, f"gpu_{k}.log"), "a")
    procs.append(subprocess.Popen([sys.executable, "scripts/phase4.py", "gpu", "--out", OUT,
                                   "--shard", str(k), "--nshards", str(max(1, NGPU))],
                                  stdout=log, stderr=subprocess.STDOUT, env=env))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(60)
    if time.time() - t0 > SESSION_HOURS * 3600:
        print("Session time limit reached: stopping the GPU work (finished items are saved).")
        for p in procs:
            if p.poll() is None:
                p.terminate()
        for p in procs:
            try:
                p.wait(timeout=120)
            except subprocess.TimeoutExpired:
                p.kill()
        break
    if int(time.time() - t0) % 1800 < 60:
        for k in range(len(procs)):
            tail = open(os.path.join(OUT, f"gpu_{k}.log")).read().splitlines()[-1:]
            print(time.strftime("%H:%M"), f"GPU {k}:", tail[0] if tail else "")
print("exit codes:", [p.returncode for p in procs])
for k in range(len(procs)):
    if procs[k].returncode not in (0, None, -15):
        print(f"--- GPU {k} log tail ---"); print("\n".join(open(os.path.join(OUT, f'gpu_{k}.log')).read().splitlines()[-30:]))

In [ ]:
# 7. Progress check. The confirmatory analysis runs only when ALL work is finished.
sys.path.insert(0, "scripts")
import importlib, phase4 as P
from pathlib import Path
importlib.reload(P)
OUTP = Path(OUT)
prep = P._prep(OUTP)
vocab = P._vocab(OUTP)
n = lambda stem: len(P.read_all(OUTP, stem))
want = {"captions_llava_coco": len(prep["cal_a"]) + len(prep["cal_b"]) + len(prep["test"]),
        "captions_llava_amber": len(prep["amber"]), "captions_qwen": len(prep["test"]), "captions_internvl": len(prep["test"])}
progress = {k: (n(k), v) for k, v in want.items()}
captions_done = all(d >= v for d, v in progress.values())
if captions_done:
    progress["owl"] = (n("owl"), len(P.owl_tasks(OUTP)))
    progress["swap"] = (n("swap"), len(P.claims_by_image(vocab, P.read_all(OUTP, "captions_llava_coco"))))
else:
    progress["owl"] = (n("owl"), None)
    progress["swap"] = (n("swap"), None)
for k, (d, v) in progress.items():
    print(f"{k:22s} {d:6d} / {v if v is not None else '(after captions)':>6}  " + (f"{100 * d / v:5.1f}%" if v else ""))
COMPLETE = captions_done and all(v is not None and d >= v for d, v in progress.values())
if COMPLETE:
    print("\nCOMPLETE - running the confirmatory analysis (CPU, ~20-40 min)")
    !python scripts/phase4.py analyze --out "$OUT"
    from IPython.display import Markdown, display
    display(Markdown(open(f"{OUT}/phase4_report.md").read()))
else:
    print("\nNOT COMPLETE - run another session: Add Input -> Your Work -> this notebook (latest version) -> Save & Run All.")

In [ ]:
# 8. Package results (no images). Made every session; the final one includes the report.
import zipfile
zpath = os.path.join(OUT, "phase4_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{OUT}/*.jsonl") + glob.glob(f"{OUT}/*.log") + [f"{OUT}/prep.json", f"{OUT}/phase4_results.json", f"{OUT}/phase4_report.md"]:
        if os.path.exists(p):
            z.write(p, os.path.basename(p))
    z.writestr("environment.txt", f"torch {torch.__version__}\ntransformers {transformers.__version__}\n"
               + subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("Saved:", zpath, round(os.path.getsize(zpath) / 1e6, 1), "MB", "| COMPLETE" if COMPLETE else "| partial - run another session")